## Load Environment Variables

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

mongodb_connection_string = os.getenv('MONGODB_CONNECTION_STRING')
gemini_api_keys = os.getenv('GEMINI_API_KEYS')

## Identify Data Source

In [1]:
data_source="https://investors.mongodb.com/node/12236/pdf"

## Prepare The Data

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(data_source)

data = loader.load()

/home/busycaesar/projects/personal/rag/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Sanitize The Data

In [3]:
sanitized_data = []

for page in data:
  if page.page_content and len(page.page_content.strip()) > 0:
    sanitized_data.append(page)

### Divide Data into Chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)

documents = text_splitter.split_documents(data)

### Convert Data into Embeddings

In [5]:
# Get the Embedding Model Instance
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1", trust_remote_code=True)

# Prepare data to store into vector database
docs_to_insert = [{
    "text": doc.page_content,
    "embedding": embedding_model.encode(doc.page_content).tolist()
} for doc in documents]

<All keys matched successfully>


## Store The Data

In [ ]:
from pymongo import MongoClient

database_name=
collection_name=

mongodb_client = MongoClient(mongodb_connection_string)

mongodb_collection = mongodb_client["rag_db"]["embeddings"]

mongodb_collection.insert_many(docs_to_insert)

## Create cosine search index
from pymongo.operations import SearchIndexModel

vector_index = "vector_index"

search_index_model = SearchIndexModel(
    definition = {
      "fields": [
        {
          "type": "vector",
          "numDimensions": 768, # 768 Vector dimensions
          "path": "embedding",
          "similarity": "cosine"
        }
      ]
    },
    name = vector_index,
    type = "vectorSearch"
)

mongodb_collection.create_search_index(model=search_index_model)

'vector_index'

## Retrieve The Data

In [7]:
user_query = "What are MongoDB's latest AI announcements?"

### Convert Query into Embeddings

In [8]:
query_embedding = embedding_model.encode(user_query).tolist()

### Fetch Relevant Chunks

In [9]:
pipeline = [
  {
    "$vectorSearch": {
      "index": vector_index,
      "queryVector": query_embedding,
      "path": "embedding",
      "exact": True,
      "limit": 3 # Fetch top three relevant chunks.
    }
  },
  {
    "$project": {
      "_id": 0,
      "text": 1
    }
  }
]

results = list(mongodb_collection.aggregate(pipeline))

relevant_chunks = " ".join([doc["text"] for doc in results])

## Generate Response

In [ ]:
# Create prompt template
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
  input_variables=["prompt", "relevant_chunk_of_data"], 
  template=
    """
    Use the following pieces of context to answer the question at the end.

    Context: {relevant_chunk_of_data}

    User's Question: {prompt}
    """
)

from langchain_google_genai import ChatGoogleGenerativeAI

generation_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", google_api_key=gemini_api_keys)

chain = prompt_template | generation_model

response = chain.invoke({
  "prompt": user_query,
  "relevant_chunk_of_data": relevant_chunks
})

print(response.content)

MongoDB's latest AI announcement is the **MongoDB AI Applications Program (MAAP)**.
